In [ ]:
'''

Locomotion speed visualisation
--------------------------------

0) Plot path traces, as before in `Requested_plots.ipynb` with figure_per_trial, with the confidence colourmap

i) Separately, plot x and y position tracks and check for abnormalities. 

ii) Plot confiedence on x,y position tracks to allow for inspection alongside the position tracks for abnormalities.

iii) Calculate and return the SD and distribution for confidence values for each point in the path traces.

iv) Construct a locomotion speed plot, with additional featues based on the `plot_session_traces` function in `Nosepoke_outbound_activations.ipynb`, excluding the nosepoke traces themselves

v) Using the SD and distribution for confidence values, manually set parameter for threshold to drop any points with confidence values outside realistic range.

vi) Re-plot path traces, label all dropped points in one colour and all remaining points in another colour, to allow for inspection of the effect of the thresholding. Include a toggle to show/hide the dropped points. Add toggle to set all points with confidence below a certain value to be set to dropped points colour.

vii) Replot x and y and confidence tracks, with the same toggles as above.

viii) Create new object/dataset, applying `movement`'s interpolation filter. Make sure that all relevant args for interpolation filter are exposed to the user in the function itself.

ix) Reproduce path traces, x and y tracks, confidence tracks, as well as locomotion speed plot,  with the new interpolated dataset. Include toggles to show/hide dropped points and to set all points with confidence below a certain value to be set to dropped points colour.



'''

# Locomotion speed visualisation

QC pass over the DLC pose: raw path traces through to a confidence-thresholded,
interpolated dataset (full plan in the first cell). Two knobs, set in the setup cell,
drive everything:

- `CENTROID_POINTS`: keypoints averaged into the tracked point (a single-item list tracks one keypoint).
- `USE_CM`: pixels (False) or cm (True, via the calibration below).

In [ ]:
# Setup | imports + the two top-level knobs the whole notebook reads.
import pathlib
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import xarray as xr

# data-conduit Q_C glue: the configured DataStructure, the training spec/filter, and the
# pose-slicing + centroid-keypoint helpers shared across the qc notebooks.
from data_conduit.qc import (
    qc_datastructure,
    training_spec,
    training_session_names,
    filter_trials,
    slice_pose_for_trial,
    DEFAULT_CENTROID_POINTS,
    TRAINING_FIRST_MID_LAST_DETAIL,
    TRAINING_PHASE_LABELS,
)
from data_conduit.qc.path_plots import _first_frame_for_session   # first arena frame per session

# movement (neuroinformatics) supplies the kinematics (speed) and filtering (confidence
# drop + interpolation) that run directly on the xarray pose arrays.
from movement.plots import plot_centroid_trajectory
import movement.kinematics as kin
import movement.filtering as mfilt

# Training data root. The first/mid/last sessions live under BonsaiOutput/Training; point
# this at whichever external drive holds them on the machine you run this on (the other qc
# notebooks use .../Elements/... or .../Elements1/...).
ROOT = pathlib.Path('/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training')

# --- The two knobs everything below reads --------------------------------------------
# CENTROID_POINTS: DLC keypoints averaged into the tracked point. Full default = robust
# body centroid; a single-item list tracks ONE keypoint. Every figure and the speed /
# interpolation steps use exactly this set.
CENTROID_POINTS = list(DEFAULT_CENTROID_POINTS)   # ('nose', 'lear', 'rear', 'body', 'tailbase')

# USE_CM: False -> positions in px, speed in px/s (no calibration needed).
#         True  -> positions in cm, speed in cm/s, via the per-session arena-circle scale.
USE_CM = False

# Per-session video subfolder holding the .avi the arena frame is read from.
VIDEO_SUBDIR = 'UndistortedVideoData'

In [ ]:
# Session selection + load (mirrors the Requested_plots / Nosepoke setup).
# Load ONLY the first / mid / last training day per mouse (TRAINING_FIRST_MID_LAST_DETAIL),
# matched by Bonsai session name, then apply the per-session trial filter. `result` carries
# the concatenated `dlc:position` (Time x keypoints x space) and `dlc:confidence`
# (Time x keypoints) arrays, each with a per-frame `session` coordinate on the Time axis.
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)
spec['phase'] = spec['training_day'].map(TRAINING_PHASE_LABELS)      # 1->first, 2->mid, 3->last

datastructure = qc_datastructure(
    root=ROOT,
    depth=2,
    level_names=('mouseID', 'day'),
    streams=('events', 'nosepoke', 'soundcard', 'session_settings', 'video', 'dlc'),
    include=training_session_names(spec),
)
result = datastructure.load()
trials_filtered = filter_trials(result['trials'], spec)

position = result['dlc:position']       # Time x keypoints x space  (session coord on Time)
confidence = result['dlc:confidence']   # Time x keypoints          (session coord on Time)
trials_filtered

## Calibration | pixel to cm scale

Fits the arena floor circle per session (same as `Requested_plots`) to get `cm_per_px`. The
grid overlays each fit for a visual check. Only applied when `USE_CM` is True.

In [ ]:
# Calibration functions (verbatim from Requested_plots' calibration cell, plus the
# fault-tolerant frame reader). `_safe_first_frame` retries the external-drive video read a
# few times before falling back to a blank background, so one flaky read can't abort a grid.
import cv2
from matplotlib.patches import Circle

ARENA_CM = 90.0   # diameter of the arena floor circle in cm


def _safe_first_frame(root, session_id, video_subdir, cache, *, retries=2, backoff=0.5):
    """Fault-tolerant `_first_frame_for_session`.

    The training videos sit on an external USB drive, and reading them under load
    intermittently fails (usually OSError('Could not load meta information') when the
    ffmpeg metadata read times out, but imageio/ffmpeg can raise other types too) even
    though the file itself is fine. Retry a few times, then fall back to None (blank
    background) with a warning, so one flaky read can't abort the whole grid. A background
    frame is decorative, hence the broad catch. Cached per session.

    Parameters
    ----------
    root : str | pathlib.Path
        Data root the DataStructure was loaded from.
    session_id : str
        Session folder name (the `session` tag on the trials table).
    video_subdir : str
        Per-session video subfolder (e.g. 'UndistortedVideoData').
    cache : dict
        Reused across calls to memoise frames (and failures) by session id.
    retries : int
        Extra attempts after the first before giving up. Default 2.
    backoff : float
        Seconds to sleep between attempts. Default 0.5.

    Returns
    -------
    numpy.ndarray | None
        The first frame (H x W x 3), or None if every read failed.
    """
    if session_id in cache:
        return cache[session_id]
    frame = None
    for attempt in range(retries + 1):
        try:
            frame = _first_frame_for_session(root, session_id, video_subdir, {})   # fresh dict -> force read
            break
        except Exception as exc:   # any read failure -> blank background, never abort the figure
            if attempt < retries:
                time.sleep(backoff)
                continue
            detail = (str(exc).strip().splitlines() or [''])[0][:100]
            print(f"warning: frame read failed for session {session_id!r} after "
                  f"{retries + 1} tries ({exc.__class__.__name__}: {detail}); "
                  f"drawing that panel without a background.")
    cache[session_id] = frame
    return frame


def fit_arena_circle(frame, floor_frac=0.7):
    """Least-squares circle fit to the bright arena floor.

    Parameters
    ----------
    frame : numpy.ndarray
        RGB (or grayscale) video frame.
    floor_frac : float
        Brightness threshold (0-1) between the dark surround and the bright floor; 0.7
        isolates the floor from the dimmer nose-poke ring. Default 0.7.

    Returns
    -------
    tuple | None
        (cx, cy, R, resid_px) of the fitted circle, or None if no contour was found.
    """
    g = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY) if frame.ndim == 3 else frame
    lo, hi = np.percentile(g, 5), np.percentile(g, 85)
    _, th = cv2.threshold(g, int(lo + floor_frac * (hi - lo)), 255, cv2.THRESH_BINARY)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, k)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea).reshape(-1, 2).astype(float)
    x, y = c[:, 0], c[:, 1]
    D, E, F = np.linalg.lstsq(np.c_[x, y, np.ones_like(x)], -(x**2 + y**2), rcond=None)[0]
    cx, cy = -D / 2, -E / 2
    R = float(np.sqrt(cx * cx + cy * cy - F))
    resid = float(np.abs(np.hypot(x - cx, y - cy) - R).mean())
    return cx, cy, R, resid


def calibrate_session(root, session, *, arena_cm=ARENA_CM,
                      video_subdir=None, manual_points=None, cache=None):
    """Pixel -> cm calibration for ONE session.

    Parameters
    ----------
    root : str | pathlib.Path
        Data root the DataStructure was loaded from.
    session : str
        Session folder name.
    arena_cm : float
        Real diameter of the arena floor circle in cm. Default ARENA_CM.
    video_subdir : str | None
        Per-session video subfolder; defaults to the notebook's VIDEO_SUBDIR.
    manual_points : tuple | None
        ((x1, y1), (x2, y2)) hand-labelled arena-edge points `arena_cm` apart; overrides
        the automatic floor-circle fit.
    cache : dict | None
        Frame cache passed to `_safe_first_frame`.

    Returns
    -------
    dict
        Calibration record: session, frame, method, cm_per_px, px_per_cm, diameter_px,
        the two ruler endpoints, a residual fraction, and an `ok` flag.
    """
    if video_subdir is None:
        video_subdir = VIDEO_SUBDIR
    frame = _safe_first_frame(root, session, video_subdir, {} if cache is None else cache)
    cal = {'session': session, 'frame': frame, 'ok': False, 'method': None}
    if manual_points is not None:
        (x1, y1), (x2, y2) = manual_points
        d = float(np.hypot(x2 - x1, y2 - y1))
        cal.update(p1=(x1, y1), p2=(x2, y2), diameter_px=d, method='manual',
                   cm_per_px=arena_cm / d, px_per_cm=d / arena_cm,
                   residual_frac=np.nan, ok=d > 0)
        return cal
    if frame is None:
        return cal                                   # unreadable frame -> flagged, no crash
    fit = fit_arena_circle(frame)
    if fit is None:
        return cal
    cx, cy, R, resid = fit
    cal.update(cx=cx, cy=cy, R=R, p1=(cx - R, cy), p2=(cx + R, cy), diameter_px=2 * R,
               method='circle', cm_per_px=arena_cm / (2 * R), px_per_cm=2 * R / arena_cm,
               residual_px=resid, residual_frac=resid / R, ok=(resid / R < 0.05))
    return cal


def show_calibration(cal, ax, *, ruler_color='cyan'):
    """Overlay the fitted circle + the two ruler endpoints on the session frame.

    Parameters
    ----------
    cal : dict
        A record from `calibrate_session`.
    ax : matplotlib.axes.Axes
        Axis to draw on.
    ruler_color : str
        Colour of the ruler line and its endpoints. Default 'cyan'.

    Returns
    -------
    None
    """
    if cal.get('frame') is not None:
        ax.imshow(cal['frame'], origin='upper')
    ax.set_xticks([]); ax.set_yticks([])
    if 'p1' not in cal:
        ax.set_title(f"{cal['session']}\n(fit failed)", fontsize=7, color='red')
        return
    if 'R' in cal:
        ax.add_patch(Circle((cal['cx'], cal['cy']), cal['R'], fill=False, ec='red', lw=1.5, zorder=3))
    (x1, y1), (x2, y2) = cal['p1'], cal['p2']
    ax.plot([x1, x2], [y1, y2], color=ruler_color, lw=1.5, zorder=4)                # the ruler
    ax.scatter([x1, x2], [y1, y2], c=ruler_color, s=45, ec='k', lw=0.6, zorder=5)   # its 2 endpoints
    res = '' if pd.isna(cal.get('residual_frac')) else f"  res {cal['residual_frac'] * 100:.1f}%"
    ax.set_title(f"{cal['session']}\nD={cal['diameter_px']:.0f}px  {cal['cm_per_px']:.4f} cm/px"
                 f"{res}{'' if cal['ok'] else '  !!'}", fontsize=6.5,
                 color=('black' if cal['ok'] else 'red'))


def calibration_grid_by_mouse(root, sess_meta, *, arena_cm=ARENA_CM, manual_points=None,
                              mouse_column='mouseID', order_col='training_day'):
    """Fit EVERY session and lay the confirmation overlays out (columns = mouse, rows = session).

    Parameters
    ----------
    root : str | pathlib.Path
        Data root the DataStructure was loaded from.
    sess_meta : pandas.DataFrame
        One row per session, indexed by session id, with `mouse_column` and `order_col`.
    arena_cm : float
        Arena floor diameter in cm. Default ARENA_CM.
    manual_points : dict | None
        {session: ((x1, y1), (x2, y2))} overrides for sessions whose auto-fit fails.
    mouse_column : str
        Column holding the mouse id (grid columns). Default 'mouseID'.
    order_col : str
        Column ordering each mouse's sessions down the rows. Default 'training_day'.

    Returns
    -------
    tuple
        (fig, session_scale DataFrame) where session_scale has session, mouse, order_col,
        cm_per_px, residual_frac and ok.
    """
    manual_points = manual_points or {}
    mice = sorted(sess_meta[mouse_column].unique())
    per_mouse = {m: list(sess_meta[sess_meta[mouse_column] == m].sort_values(order_col).index)
                 for m in mice}
    nrows = max(len(v) for v in per_mouse.values())
    fig, axes = plt.subplots(nrows, len(mice), figsize=(3.1 * len(mice), 2.8 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    cache, rows = {}, []
    for ci, m in enumerate(mice):
        for r, s in enumerate(per_mouse[m]):
            ax = axes[r][ci]; ax.axis('on')
            cal = calibrate_session(root, s, arena_cm=arena_cm,
                                    manual_points=manual_points.get(s), cache=cache)
            show_calibration(cal, ax)
            rows.append({'session': s, mouse_column: m, order_col: sess_meta.loc[s, order_col],
                         'cm_per_px': cal.get('cm_per_px'), 'residual_frac': cal.get('residual_frac'),
                         'ok': cal.get('ok')})
        axes[0][ci].annotate(m, xy=(0.5, 1.0), xytext=(0, 22), textcoords='offset points',
                             xycoords='axes fraction', ha='center', va='bottom',
                             fontsize=11, weight='bold')
    fig.suptitle('Per-session arena calibration - columns = mouse, rows = session  '
                 f'(red = fitted floor circle; cyan = {arena_cm:g} cm ruler)', y=1.0)
    fig.tight_layout()
    return fig, pd.DataFrame(rows)


def plot_scale_distribution(session_scale, *, mouse_column='mouseID'):
    """Distribution of cm/px across sessions.

    Parameters
    ----------
    session_scale : pandas.DataFrame
        The table returned by `calibration_grid_by_mouse`.
    mouse_column : str
        Column holding the mouse id. Default 'mouseID'.

    Returns
    -------
    matplotlib.figure.Figure
        Left panel: one dot per session grouped by mouse (black bar = mouse mean).
        Right panel: histogram of cm/px across all sessions with the median marked.
    """
    good = session_scale.dropna(subset=['cm_per_px'])
    mice = sorted(good[mouse_column].unique())
    cmap = {m: f'C{i % 10}' for i, m in enumerate(mice)}
    rng = np.random.default_rng(0)
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={'width_ratios': [2, 1]})
    for xi, m in enumerate(mice):
        v = good.loc[good[mouse_column] == m, 'cm_per_px'].to_numpy()
        axL.scatter(np.full(len(v), xi) + (rng.random(len(v)) - 0.5) * 0.25, v,
                    color=cmap[m], s=30, alpha=0.85, zorder=3)
        axL.plot([xi - 0.22, xi + 0.22], [v.mean(), v.mean()], color='k', lw=2, zorder=4)
    axL.set_xticks(range(len(mice))); axL.set_xticklabels(mice, rotation=45, ha='right', fontsize=8)
    axL.set_ylabel('cm_per_px'); axL.set_title('cm/px per session  (one dot = one session)')
    axR.hist(good['cm_per_px'], bins=15, color='grey', alpha=0.85)
    med = good['cm_per_px'].median()
    axR.axvline(med, color='crimson', ls='--', label=f'median = {med:.4f}')
    axR.set_xlabel('cm_per_px'); axR.set_title(f'distribution (n={len(good)} sessions)')
    axR.legend(fontsize=8)
    fig.tight_layout()
    return fig

In [ ]:
# Fit EVERY session, show the confirmation grid + the cm/px distribution, and build the
# session -> cm/px map used when USE_CM is True. Good fits keep their scale; failures show
# as NaN so they stand out (pass manual_points={session: ((x1,y1),(x2,y2))} to fix one).
sess_meta = (trials_filtered[['session', 'mouseID', 'training_day']]
             .drop_duplicates('session').set_index('session'))

fig_grid, session_scale = calibration_grid_by_mouse(ROOT, sess_meta)   # fits ALL sessions
plt.show()
plot_scale_distribution(session_scale)
plt.show()

# session -> cm_per_px (good fits only; NaN elsewhere so they stand out).
scale_map = (session_scale.set_index('session')['cm_per_px']
             .where(session_scale.set_index('session')['ok'].to_numpy()))
missing = scale_map.index[scale_map.isna()].tolist()
print(f"{scale_map.notna().sum()}/{len(scale_map)} sessions with a usable cm/px scale"
      + (f"; need manual_points: {missing}" if missing else "; all good."))

# Unit wiring read by every plot below. cm_per_px_for(session) scales px -> cm for that
# session; when USE_CM is False the plots stay in pixels and it is unused.
LENGTH_UNIT = 'cm' if USE_CM else 'px'
SPEED_UNIT = 'cm/s' if USE_CM else 'px/s'


def cm_per_px_for(session):
    """Arena cm per pixel for a session (from its floor-circle fit); np.nan if unusable."""
    return float(scale_map.get(session, np.nan))

## Centroid & confidence

Tracked point = mean of `CENTROID_POINTS`; its confidence = the *minimum* likelihood over
those keypoints. Warns (with a per-session breakdown) if any keypoint is NaN on some frames,
even when the centroid is still recoverable from the rest.

In [ ]:
# Build the tracked point and its confidence, and surface NaN keypoints to the user.
def _keypoint_nan_report(position_sub, *, keypoint_dim='keypoints', space_dim='space',
                         time_coord='Time', session_coord='session'):
    """Per (session, keypoint) count/fraction of frames where that keypoint is NaN.

    A keypoint is 'missing' on a frame if EITHER coordinate (x or y) is NaN. The centroid
    can still be formed from the remaining keypoints on such frames, so this is a heads-up,
    not an error.

    Parameters
    ----------
    position_sub : xarray.DataArray
        `position` already restricted to the centroid keypoints (Time x keypoints x space).
    keypoint_dim, space_dim, time_coord, session_coord : str
        Dimension / coordinate names on `position_sub`.

    Returns
    -------
    pandas.DataFrame
        Rows (session, keypoint, n_missing, n_frames, frac_missing) for keypoints with at
        least one missing frame, sorted worst-first within each session.
    """
    missing = position_sub.isnull().any(space_dim)                     # Time x keypoints (bool)
    n_missing = missing.groupby(session_coord).sum(time_coord)         # session x keypoints
    n_frames = missing.groupby(session_coord).count(time_coord)        # session x keypoints (= frames/session)
    df = (n_missing.rename('n_missing').to_dataframe().reset_index()
          .merge(n_frames.rename('n_frames').to_dataframe().reset_index(),
                 on=[session_coord, keypoint_dim]))
    df['frac_missing'] = df['n_missing'] / df['n_frames']
    return (df[df['n_missing'] > 0]
            .sort_values([session_coord, 'frac_missing'], ascending=[True, False])
            .reset_index(drop=True))


def centroid_position(position, centroid_points, *, warn=True,
                      keypoint_dim='keypoints', space_dim='space', time_coord='Time'):
    """Tracked point = mean of `centroid_points` per frame (skipping NaN keypoints).

    Parameters
    ----------
    position : xarray.DataArray
        Full pose (Time x keypoints x space) with a `session` coord on the Time axis.
    centroid_points : sequence of str
        Keypoint names to average. A single-item list yields that one keypoint.
    warn : bool
        If True, print/display a NaN-keypoint report (see `_keypoint_nan_report`) and
        count frames where the centroid is entirely undefined. Default True.
    keypoint_dim, space_dim, time_coord : str
        Dimension names on `position`.

    Returns
    -------
    xarray.DataArray
        The centroid (Time x space), carrying the same `session` coord as `position`.
    """
    sub = position.sel({keypoint_dim: list(centroid_points)})
    if warn:
        report = _keypoint_nan_report(sub, keypoint_dim=keypoint_dim, space_dim=space_dim,
                                      time_coord=time_coord)
        all_missing = int(sub.isnull().any(space_dim).all(keypoint_dim).sum())   # centroid NaN frames
        if not report.empty:
            totals = report.groupby(keypoint_dim)['n_missing'].sum()
            summary = ', '.join(f'{k}: {int(v)}' for k, v in totals.items())
            warnings.warn(
                f'NaN among CENTROID_POINTS on some frames (centroid still formed from the '
                f'rest): {summary} missing frames by keypoint. '
                f'{all_missing} frame(s) have ALL centroid keypoints missing (centroid = NaN). '
                f'Per-session breakdown displayed below.',
                stacklevel=2)
            try:
                display(report)                       # per-session detail in a notebook
            except NameError:
                print(report.to_string(index=False))
        elif all_missing:
            print(f'{all_missing} frame(s) have ALL centroid keypoints missing (centroid = NaN).')
    return sub.mean(keypoint_dim, skipna=True)


def centroid_confidence(confidence, centroid_points, *, keypoint_dim='keypoints'):
    """Per-frame confidence for the centroid = MIN DLC likelihood over its keypoints.

    Parameters
    ----------
    confidence : xarray.DataArray
        Full confidence (Time x keypoints) with a `session` coord on the Time axis.
    centroid_points : sequence of str
        The same keypoints averaged into the centroid.
    keypoint_dim : str
        Keypoint dimension name. Default 'keypoints'.

    Returns
    -------
    xarray.DataArray
        Confidence (Time,), carrying the same `session` coord as `confidence`.
    """
    return confidence.sel({keypoint_dim: list(centroid_points)}).min(keypoint_dim)


# The tracked point + its confidence, used by every figure below. Re-run this after
# changing CENTROID_POINTS. The NaN warning fires here, once, for the raw pose.
centroid = centroid_position(position, CENTROID_POINTS)          # Time x space (mean of keypoints)
centroid_conf = centroid_confidence(confidence, CENTROID_POINTS) # Time (min likelihood over keypoints)
centroid

In [ ]:
# Shared per-session infrastructure for the time-series figures (tracks + speed plot).
# The trial-context overlay (outbound/inbound spans, Start Trial events, decision poke,
# outcome marker) is the SAME idea as plot_session_traces in the Nosepoke notebook, here
# factored out so the position-track and speed figures draw identical context.
OUTCOME_STYLE = {'Success': ('#2e8b57', '^'), 'Failure': ('#d1495b', 'v'), 'Miss': ('#888888', 'o')}


def _session_slice(arr, session, *, session_coord='session', time_coord='Time'):
    """One session's entries of a Time-indexed DataArray, in Time order."""
    idx = np.flatnonzero(arr.coords[session_coord].values == session)
    return arr.isel({time_coord: idx})


def _centroid_xy(sliced, centroid_points, *, keypoint_dim='keypoints', space_dim='space'):
    """(x, y) arrays of the centroid (mean of centroid_points) per frame of a pose slice."""
    c = sliced.sel({keypoint_dim: list(centroid_points)}).mean(keypoint_dim, skipna=True)
    return c.sel({space_dim: 'x'}).values, c.sel({space_dim: 'y'}).values


def _start_trial_times(root, session, prefix='Start Trial'):
    """Seconds of raw '<prefix>' events from a session's ExperimentEvents CSV (same clock
    as Time). Split on the FIRST comma only, since some event strings contain commas."""
    matches = sorted(pathlib.Path(root).glob(f'**/{session}/ExperimentEvents/*.csv'))
    if not matches:
        return np.array([])
    lines = pathlib.Path(matches[0]).read_text().splitlines()[1:]      # drop the header row
    rows = [ln.split(',', 1) for ln in lines if ln]
    ev = pd.DataFrame(rows, columns=['Seconds', 'Value'])
    return ev.loc[ev['Value'].str.startswith(prefix), 'Seconds'].astype(float).to_numpy()


def _overlay_trial_events(ax, tr_sess, st_ev, w0, w1, *, marker_y=None):
    """Draw trial context on a windowed time axis: outbound (blue) / inbound (orange)
    spans, Start Trial events (green dotted), the decision poke (black dashed) and, if
    marker_y is given, the outcome marker at that height. Shared by the track + speed figs.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis whose x-axis is Time (seconds).
    tr_sess : pandas.DataFrame
        This session's trial rows (start_time, outbound_end_time, end_time, outcome).
    st_ev : numpy.ndarray
        Start Trial event times (seconds) for the session.
    w0, w1 : float
        Window bounds (seconds); only events/trials overlapping [w0, w1) are drawn.
    marker_y : float | None
        Height for the outcome marker; None to omit it (e.g. on stacked track panels).

    Returns
    -------
    None
    """
    for t in st_ev[(st_ev >= w0) & (st_ev < w1)]:
        ax.axvline(t, color='#2ca02c', ls=':', lw=1.0, alpha=0.85)                # raw Start Trial
    wt = tr_sess[(tr_sess['start_time'] < w1) & (tr_sess['end_time'] >= w0)]
    for _, r in wt.iterrows():
        has_in = pd.notna(r['outbound_end_time'])
        ob_end = r['outbound_end_time'] if has_in else r['end_time']
        ax.axvspan(r['start_time'], ob_end, color='#4c72b0', alpha=0.06)           # outbound leg
        ax.axvline(r['start_time'], color='#4c72b0', lw=0.8, alpha=0.6)            # outbound start
        if has_in:
            ax.axvspan(r['outbound_end_time'], r['end_time'], color='#dd8452', alpha=0.10)  # inbound leg
            ax.axvline(r['outbound_end_time'], color='#dd8452', lw=0.8, alpha=0.6)          # inbound start
        col, mk = OUTCOME_STYLE.get(r['outcome'], ('k', 'x'))
        ax.axvline(r['end_time'], color='k', ls='--', lw=0.9, alpha=0.55)          # decision poke
        if marker_y is not None:
            ax.plot(r['end_time'], marker_y, marker=mk, color=col, ms=7)           # outcome marker

## 0 | Path traces

`figure_per_trial` (from `Requested_plots`): one panel per trial, centroid points shaded by
confidence, over the arena frame. One mouse at a time.

In [ ]:
# Path-trace grid, per trial, shaded by centroid confidence (from Requested_plots).
def figure_per_trial(
        result, trials, root, mouse, *,
        group_by='training_day',                 # outer columns: 'session' or 'training_day'
        trial_number_column=None,                # panel label; defaults to match group_by
        segments=('outbound', 'inbound'),        # subcolumns
        centroid_points=DEFAULT_CENTROID_POINTS, # centroid = mean of these keypoints
        segment_cmaps=None,                      # {segment: cmap}; default per segment below
        conf_key='dlc:confidence', conf_vmin=0.0, conf_vmax=1.0,
        video_subdir='UndistortedVideoData',
        mouse_column='mouseID', session_column='session',
        pose_key='dlc:position', time_coord='Time',
        marker_size=6, figsize_per_cell=(2.3, 1.9), max_trials=None):
    """Small-multiples grid of per-trial centroid paths for ONE mouse.

    Row r of a grouping column is that grouping's r-th trial (sorted by trial number);
    groupings with fewer trials leave blank rows. Each panel is labelled with the
    grouping's own trial number, so labels stay correct even when numbers differ
    across columns (e.g. per-session trial_index vs per-day day_trial_index).
    """
    pose = result[pose_key]
    conf = result[conf_key]
    sub = trials[trials[mouse_column] == mouse]
    if sub.empty:
        raise ValueError(f'no trials for mouse {mouse!r}.')

    if trial_number_column is None:
        trial_number_column = ('day_trial_index'
                               if group_by in ('training_day', 'day') and 'day_trial_index' in sub
                               else 'trial_index')

    groupings = sorted(pd.unique(sub[group_by]))
    default_cmaps = {'outbound': 'Blues', 'inbound': 'Oranges'}
    seg_cmap = {seg: (segment_cmaps or {}).get(seg, default_cmaps.get(seg, 'viridis'))
                for seg in segments}
    per_group = {g: sub[sub[group_by] == g].sort_values(trial_number_column).reset_index(drop=True)
                 for g in groupings}
    nrows = max(len(df) for df in per_group.values())
    if max_trials is not None:
        nrows = min(nrows, max_trials)
    blocks = [(g, seg) for g in groupings for seg in segments]   # column order
    ncols = len(blocks)

    # Pre-load each session's arena frame ONCE, up front and fault-tolerantly, so the
    # heavy render loop below only reads cached arrays. (Reading these external-drive
    # videos while rendering hundreds of big panels is what triggered the intermittent
    # ffmpeg 'Could not load meta information' error.) A reference frame then fixes a
    # shared pixel extent so the whole arena is visible in every panel.
    frame_cache = {}
    for s in pd.unique(sub[session_column]):
        _safe_first_frame(root, s, video_subdir, frame_cache)
    ref = next((f for f in frame_cache.values() if f is not None), None)
    extent = ref.shape[:2] if ref is not None else None   # (H, W)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_cell[0] * ncols, figsize_per_cell[1] * nrows),
        squeeze=False,
    )
    for ci, (g, seg) in enumerate(blocks):
        gdf = per_group[g]
        for r in range(nrows):
            ax = axes[r][ci]
            if r >= len(gdf):
                ax.axis('off')
                continue
            row = gdf.iloc[r]

            frame = _safe_first_frame(root, row[session_column], video_subdir, frame_cache)
            if frame is not None:
                ax.imshow(frame, origin='upper')

            sliced = slice_pose_for_trial(pose, row, segment=seg, time_coord=time_coord)
            if sliced.sizes.get(time_coord, 0) > 0:
                movement_pose = (sliced.rename({time_coord: 'time', 'keypoints': 'keypoint'})
                                 .expand_dims({'individual': ['ind']}))
                # Shade each centroid point by the WORST (min) DLC likelihood among the
                # centroid keypoints on that frame - one bad keypoint is what drags the
                # mean, so min flags it. Same slice window -> same Time order as position,
                # so c[i] lines up with the i-th plotted centroid point. Outbound/inbound
                # keep distinct colourmaps so the segment still reads at a glance.
                conf_sliced = slice_pose_for_trial(conf, row, segment=seg, time_coord=time_coord)
                c_conf = conf_sliced.sel(keypoints=list(centroid_points)).min('keypoints').values
                plot_centroid_trajectory(
                    movement_pose, keypoints=list(centroid_points), ax=ax,
                    c=c_conf, cmap=seg_cmap[seg], vmin=conf_vmin, vmax=conf_vmax,
                    s=marker_size,
                )

            # plot_centroid_trajectory writes its own title/labels; override AFTER it.
            if extent is not None:
                ax.set_xlim(0, extent[1]); ax.set_ylim(extent[0], 0)
            else:
                ax.invert_yaxis()
            ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(''); ax.set_ylabel('')
            tnum = row[trial_number_column]
            ax.set_title(f"{g}\n{seg}" if r == 0 else '', fontsize=9)
            ax.text(0.04, 0.96, f"#{int(tnum)}" if pd.notna(tnum) else "#?",
                    transform=ax.transAxes, va='top', ha='left', fontsize=7, color='white',
                    bbox=dict(boxstyle='round', fc='black', alpha=0.45, ec='none'))

    fig.suptitle(f"{mouse}: per-trial centroid paths  (columns = {group_by} x segment)", y=1.0)
    # One DLC-likelihood colourbar per segment (each segment uses its own colourmap).
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize
    norm = Normalize(conf_vmin, conf_vmax)
    for seg in segments:
        sm = ScalarMappable(norm=norm, cmap=seg_cmap[seg])
        cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.012, pad=0.01)
        cbar.set_label(f'{seg} DLC likelihood (min over centroid keypoints)')
    fig.tight_layout()
    return fig, axes

In [ ]:
# Choose the mouse to inspect (a session/day belongs to one mouse). group_by='training_day'
# uses the tallest day for the row count; pass max_trials to cap very tall grids.
MOUSE = 'FL_M01569519'
# 'FLR_M01569521','FL_M01569519','FbR_M01569522','MR_M01569515','MbL_M01569517','MbR_M01569518'

fig, axes = figure_per_trial(
    result, trials_filtered, ROOT, MOUSE,
    group_by='training_day',
    centroid_points=CENTROID_POINTS,      # same tracked point as the rest of the notebook
    video_subdir=VIDEO_SUBDIR,
    max_trials=80,
)
plt.show()

## i, ii | x / y / confidence tracks

Continuous `x(t)`, `y(t)` and `confidence(t)` per session, windowed with trial context. A
position jump that lines up with a confidence dip is a tracking glitch, not real movement.

In [ ]:
def plot_position_tracks(centroid, centroid_conf, trials, session, *, window_s=60,
                         use_cm=None, root=None, event_prefix='Start Trial',
                         figsize=(14, 6), session_coord='session', time_coord='Time',
                         space_dim='space'):
    """Per-session x(t), y(t) and confidence(t) tracks, windowed, with trial context.

    Parameters
    ----------
    centroid : xarray.DataArray
        The tracked point (Time x space) with a `session` coord on Time.
    centroid_conf : xarray.DataArray
        Its per-frame confidence (Time,) with the same `session` coord.
    trials : pandas.DataFrame
        The (filtered) trial table; rows for `session` supply the overlay context.
    session : str
        Session id to plot.
    window_s : float
        Seconds of Time per panel. Default 60.
    use_cm : bool | None
        Positions in cm (True) or px (False). None -> the notebook's USE_CM.
    root : str | pathlib.Path | None
        Data root for reading Start Trial events. None -> the notebook's ROOT.
    event_prefix : str
        Event-name prefix counted as a trial start. Default 'Start Trial'.
    figsize : tuple
        Size of each 3-row window figure.
    session_coord, time_coord, space_dim : str
        Coordinate / dimension names.

    Returns
    -------
    None
        Draws one stacked (x, y, confidence) figure per time window.
    """
    if use_cm is None:
        use_cm = USE_CM
    if root is None:
        root = ROOT
    c = _session_slice(centroid, session, session_coord=session_coord, time_coord=time_coord)
    q = _session_slice(centroid_conf, session, session_coord=session_coord, time_coord=time_coord)
    if c.sizes.get(time_coord, 0) == 0:
        print(f'no centroid frames for session {session!r}.')
        return
    t = c.coords[time_coord].values
    x = c.sel({space_dim: 'x'}).values.astype(float)
    y = c.sel({space_dim: 'y'}).values.astype(float)
    conf = q.values.astype(float)
    unit = 'px'
    if use_cm:                                     # scale px -> cm by this session's calibration
        s = cm_per_px_for(session)
        if np.isnan(s):
            print(f'session {session!r} has no usable cm/px scale; showing pixels.')
        else:
            x, y, unit = x * s, y * s, 'cm'
    tr_sess = trials[trials['session'] == session]
    st_ev = _start_trial_times(root, session, event_prefix)
    mouse = tr_sess['mouseID'].iloc[0] if len(tr_sess) else ''
    tmin, tmax = float(np.nanmin(t)), float(np.nanmax(t))
    for w0 in np.arange(tmin, tmax, window_s):     # one figure per window
        w1 = w0 + window_s
        m = (t >= w0) & (t < w1)
        fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
        axes[0].plot(t[m], x[m], lw=0.8, color='#4c72b0'); axes[0].set_ylabel(f'x ({unit})')
        axes[1].plot(t[m], y[m], lw=0.8, color='#dd8452'); axes[1].set_ylabel(f'y ({unit})')
        axes[2].plot(t[m], conf[m], lw=0.8, color='0.35')
        axes[2].set_ylabel('confidence'); axes[2].set_ylim(0, 1.02)
        for ax in axes:                            # same trial context on every row
            _overlay_trial_events(ax, tr_sess, st_ev, w0, w1, marker_y=None)
            ax.set_xlim(w0, w1)
        axes[-1].set_xlabel('Time (s)')
        axes[0].set_title(f'{mouse}  {session}   |   {w0:.0f}-{w1:.0f}s   '
                          f'(blue span=outbound, orange=inbound, green:=Start Trial, black--=poke)',
                          fontsize=8)
        fig.tight_layout(); plt.show()

In [ ]:
# Pick a session to inspect. Any session id in trials_filtered works; this defaults to the
# first session of the mouse chosen above. Lengthen window_s for fewer, wider panels.
SESSION = trials_filtered.loc[trials_filtered['mouseID'] == MOUSE, 'session'].iloc[0]

plot_position_tracks(centroid, centroid_conf, trials_filtered, SESSION, window_s=60)

## iii | Confidence SD & distribution

Mean, SD and low percentiles of centroid confidence over the in-trial points, plus the
histogram. Read the 1 to 5% tail to pick the drop threshold in step v.

In [ ]:
def path_confidence_values(centroid_conf, trials, *, segment='trial', time_coord='Time'):
    """Centroid confidence at every in-trial frame (the points drawn in the path traces).

    Parameters
    ----------
    centroid_conf : xarray.DataArray
        Per-frame centroid confidence (Time,) with a `session` coord on Time.
    trials : pandas.DataFrame
        The (filtered) trial table.
    segment : str
        Which window per trial: 'trial' (whole), 'outbound' or 'inbound'. Default 'trial'.
    time_coord : str
        Time dimension name. Default 'Time'.

    Returns
    -------
    pandas.DataFrame
        Long table [session, mouseID, trial_index, Time, confidence], one row per in-trial
        frame across all trials.
    """
    parts = []
    for _, row in trials.iterrows():
        sl = slice_pose_for_trial(centroid_conf, row, segment=segment, time_coord=time_coord)
        if sl.sizes.get(time_coord, 0) == 0:
            continue
        df = sl.to_dataframe(name='confidence').reset_index()
        df['mouseID'] = row['mouseID']
        df['trial_index'] = row['trial_index']
        parts.append(df[['session', 'mouseID', 'trial_index', time_coord, 'confidence']])
    cols = ['session', 'mouseID', 'trial_index', time_coord, 'confidence']
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=cols)


def summarise_path_confidence(values, *, bins=40):
    """SD + distribution of centroid confidence over the path-trace points.

    Parameters
    ----------
    values : pandas.DataFrame
        Output of `path_confidence_values`.
    bins : int
        Histogram bin count. Default 40.

    Returns
    -------
    pandas.DataFrame
        Per-session summary (mean, sd, p05, n). Also prints the overall mean/SD and low
        percentiles and draws the histogram + per-session SD bars.
    """
    v = values['confidence'].to_numpy()
    v = v[~np.isnan(v)]
    overall_mean, overall_sd = float(np.mean(v)), float(np.std(v))
    pct = np.percentile(v, [1, 5, 10, 25, 50])
    print(f'centroid confidence over {len(v)} in-trial points: '
          f'mean={overall_mean:.3f}, SD={overall_sd:.3f}')
    print(f'  percentiles  1%={pct[0]:.3f}  5%={pct[1]:.3f}  '
          f'10%={pct[2]:.3f}  25%={pct[3]:.3f}  50%={pct[4]:.3f}')
    per_session = (values.groupby('session')['confidence']
                   .agg(mean='mean', sd='std', p05=lambda s: s.quantile(0.05), n='size')
                   .reset_index())
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios': [2, 1]})
    axL.hist(v, bins=bins, color='grey', alpha=0.85)
    for p, lab in zip(pct[[1, 2]], ['1%', '5%']):
        axL.axvline(p, ls='--', color='crimson')
        axL.text(p, axL.get_ylim()[1] * 0.9, lab, color='crimson', fontsize=8)
    axL.set_xlabel('centroid confidence (min over keypoints)'); axL.set_ylabel('in-trial frames')
    axL.set_title(f'distribution (mean={overall_mean:.3f}, SD={overall_sd:.3f})')
    axR.bar(range(len(per_session)), per_session['sd'], color='#4c72b0')
    axR.set_xticks(range(len(per_session)))
    axR.set_xticklabels(per_session['session'], rotation=90, fontsize=5)
    axR.set_ylabel('per-session SD'); axR.set_title('confidence SD by session')
    fig.tight_layout(); plt.show()
    return per_session

In [ ]:
# Confidence over every in-trial centroid point, then its SD / distribution. Use the low
# percentiles printed here to set CONF_THRESHOLD in step v.
path_conf = path_confidence_values(centroid_conf, trials_filtered, segment='trial')
conf_summary = summarise_path_confidence(path_conf)
conf_summary

## iv | Locomotion speed

Speed of the tracked point (`compute_speed`), windowed with the same trial context as the
tracks (no nosepoke rows). `smooth_window` optionally rolling-averages the trace.

In [ ]:
def plot_locomotion_speed(centroid, trials, session, *, window_s=60, use_cm=None, root=None,
                          smooth_window=None, event_prefix='Start Trial', figsize=(14, 3.4),
                          session_coord='session', time_coord='Time', space_dim='space'):
    """Per-session locomotion speed, windowed, with trial context (no nosepoke rows).

    Parameters
    ----------
    centroid : xarray.DataArray
        The tracked point (Time x space) with a `session` coord on Time.
    trials : pandas.DataFrame
        The (filtered) trial table; rows for `session` supply the overlay context.
    session : str
        Session id to plot.
    window_s : float
        Seconds of Time per panel. Default 60.
    use_cm : bool | None
        Speed in cm/s (True) or px/s (False). None -> the notebook's USE_CM.
    root : str | pathlib.Path | None
        Data root for reading Start Trial events. None -> the notebook's ROOT.
    smooth_window : int | None
        Rolling-mean window in frames applied to speed (None/<=1 = raw). Default None.
    event_prefix : str
        Event-name prefix counted as a trial start. Default 'Start Trial'.
    figsize : tuple
        Size of each window figure.
    session_coord, time_coord, space_dim : str
        Coordinate / dimension names.

    Returns
    -------
    None
        Draws one speed figure per time window.
    """
    if use_cm is None:
        use_cm = USE_CM
    if root is None:
        root = ROOT
    c = _session_slice(centroid, session, session_coord=session_coord, time_coord=time_coord)
    if c.sizes.get(time_coord, 0) == 0:
        print(f'no centroid frames for session {session!r}.')
        return
    t = c.coords[time_coord].values
    speed = kin.compute_speed(c.rename({time_coord: 'time'})).values.astype(float)   # px/s
    unit = 'px/s'
    if use_cm:                                     # scale px/s -> cm/s by this session's calibration
        s = cm_per_px_for(session)
        if np.isnan(s):
            print(f'session {session!r} has no usable cm/px scale; showing px/s.')
        else:
            speed, unit = speed * s, 'cm/s'
    if smooth_window and smooth_window > 1:         # optional jitter damping
        speed = pd.Series(speed).rolling(smooth_window, center=True, min_periods=1).mean().to_numpy()
    tr_sess = trials[trials['session'] == session]
    st_ev = _start_trial_times(root, session, event_prefix)
    mouse = tr_sess['mouseID'].iloc[0] if len(tr_sess) else ''
    finite = speed[np.isfinite(speed)]
    ymax = float(finite.max()) if finite.size else 1.0
    tmin, tmax = float(np.nanmin(t)), float(np.nanmax(t))
    for w0 in np.arange(tmin, tmax, window_s):     # one figure per window
        w1 = w0 + window_s
        m = (t >= w0) & (t < w1)
        fig, ax = plt.subplots(figsize=figsize)
        ax.plot(t[m], speed[m], lw=0.8, color='#6a5acd')
        _overlay_trial_events(ax, tr_sess, st_ev, w0, w1, marker_y=ymax * 1.02)
        ax.set_xlim(w0, w1); ax.set_ylim(0, ymax * 1.1)
        ax.set_xlabel('Time (s)'); ax.set_ylabel(f'speed ({unit})')
        sm = f'  smooth={smooth_window}f' if smooth_window and smooth_window > 1 else ''
        ax.set_title(f'{mouse}  {session}   |   {w0:.0f}-{w1:.0f}s{sm}   '
                     f'(blue span=outbound, orange=inbound, black--=poke, marker=outcome above)',
                     fontsize=8)
        fig.tight_layout(); plt.show()

In [ ]:
# Speed for the session chosen above. smooth_window rolls over N frames; None = raw.
plot_locomotion_speed(centroid, trials_filtered, SESSION, window_s=60, smooth_window=None)

## v | Drop threshold

Set `CONF_THRESHOLD` from the step iii tail. Points below it are treated as *dropped* by the
flagged replots and the interpolation step.

In [ ]:
# Set from the step iii percentiles. Points with centroid confidence below this are dropped.
CONF_THRESHOLD = 0.6

dropped = centroid_conf < CONF_THRESHOLD                  # Time (bool), session coord preserved
n_drop, n_tot = int(dropped.sum()), int(dropped.size)
print(f'{n_drop}/{n_tot} centroid points ({100 * n_drop / max(n_tot, 1):.1f}%) below '
      f'confidence {CONF_THRESHOLD} -> flagged as dropped.')

## vi | Path traces: kept vs dropped

Recolours the path traces: kept (blue) vs dropped (red). Toggles: `show_dropped` (hide the
red points) and `conf_cutoff` (preview a different threshold live).

In [ ]:
def figure_per_trial_flagged(pose, conf, trials, root, mouse, *,
                             conf_threshold=None, show_dropped=True, conf_cutoff=None,
                             centroid_points=None, group_by='training_day',
                             trial_number_column=None, segments=('outbound', 'inbound'),
                             kept_color='#1f77b4', dropped_color='#d62728', video_subdir=None,
                             mouse_column='mouseID', session_column='session',
                             time_coord='Time', marker_size=9, figsize_per_cell=(2.3, 1.9),
                             max_trials=None):
    """Per-trial path traces with each centroid point coloured kept vs dropped.

    Layout mirrors `figure_per_trial` (columns = group_by x segment, rows = trial slot). A
    point is 'dropped' when its centroid confidence (min over centroid keypoints) is below
    the effective cutoff: `conf_cutoff` if given, else `conf_threshold`, else CONF_THRESHOLD.

    Parameters
    ----------
    pose : xarray.DataArray
        Pose (Time x keypoints x space) with a `session` coord on Time (raw or interpolated).
    conf : xarray.DataArray
        Matching confidence (Time x keypoints) with the same `session` coord.
    trials : pandas.DataFrame
        The (filtered) trial table.
    root : str | pathlib.Path
        Data root for the arena background frames.
    mouse : str
        Mouse id to draw (one mouse per figure).
    conf_threshold : float | None
        Stored drop threshold; None -> CONF_THRESHOLD.
    show_dropped : bool
        Draw the dropped (red) points. False hides them. Default True.
    conf_cutoff : float | None
        Live override: recolour points below this as dropped, ignoring conf_threshold.
    centroid_points : sequence | None
        Keypoints forming the centroid; None -> CENTROID_POINTS.
    group_by, trial_number_column, segments : see `figure_per_trial`.
    kept_color, dropped_color : str
        Colours for surviving vs dropped points.
    video_subdir : str | None
        Per-session video subfolder; None -> VIDEO_SUBDIR.
    mouse_column, session_column, time_coord : str
        Column / dimension names.
    marker_size : float
        Scatter point size. Default 9.
    figsize_per_cell : tuple
        Size of each panel.
    max_trials : int | None
        Cap on rows (tallest grouping). None = no cap.

    Returns
    -------
    tuple
        (fig, axes).
    """
    if centroid_points is None:
        centroid_points = CENTROID_POINTS
    if video_subdir is None:
        video_subdir = VIDEO_SUBDIR
    cutoff = (conf_cutoff if conf_cutoff is not None
              else conf_threshold if conf_threshold is not None else CONF_THRESHOLD)
    sub = trials[trials[mouse_column] == mouse]
    if sub.empty:
        raise ValueError(f'no trials for mouse {mouse!r}.')
    if trial_number_column is None:
        trial_number_column = ('day_trial_index'
                               if group_by in ('training_day', 'day') and 'day_trial_index' in sub
                               else 'trial_index')
    groupings = sorted(pd.unique(sub[group_by]))
    per_group = {g: sub[sub[group_by] == g].sort_values(trial_number_column).reset_index(drop=True)
                 for g in groupings}
    nrows = max(len(df) for df in per_group.values())
    if max_trials is not None:
        nrows = min(nrows, max_trials)
    blocks = [(g, seg) for g in groupings for seg in segments]
    ncols = len(blocks)

    frame_cache = {}                               # pre-load arena frames once, fault-tolerantly
    for s in pd.unique(sub[session_column]):
        _safe_first_frame(root, s, video_subdir, frame_cache)
    ref = next((f for f in frame_cache.values() if f is not None), None)
    extent = ref.shape[:2] if ref is not None else None

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figsize_per_cell[0] * ncols, figsize_per_cell[1] * nrows),
                             squeeze=False)
    for ci, (g, seg) in enumerate(blocks):
        gdf = per_group[g]
        for r in range(nrows):
            ax = axes[r][ci]
            if r >= len(gdf):
                ax.axis('off')
                continue
            row = gdf.iloc[r]
            frame = _safe_first_frame(root, row[session_column], video_subdir, frame_cache)
            if frame is not None:
                ax.imshow(frame, origin='upper')
            sliced = slice_pose_for_trial(pose, row, segment=seg, time_coord=time_coord)
            if sliced.sizes.get(time_coord, 0) > 0:
                x, y = _centroid_xy(sliced, centroid_points)
                conf_sliced = slice_pose_for_trial(conf, row, segment=seg, time_coord=time_coord)
                c_conf = conf_sliced.sel(keypoints=list(centroid_points)).min('keypoints').values
                is_drop = c_conf < cutoff
                ax.plot(x, y, '-', color='0.6', lw=0.4, alpha=0.5, zorder=1)               # path line
                ax.scatter(x[~is_drop], y[~is_drop], s=marker_size, color=kept_color, zorder=2)
                if show_dropped and is_drop.any():
                    ax.scatter(x[is_drop], y[is_drop], s=marker_size, color=dropped_color, zorder=3)
            if extent is not None:
                ax.set_xlim(0, extent[1]); ax.set_ylim(extent[0], 0)
            else:
                ax.invert_yaxis()
            ax.set_xticks([]); ax.set_yticks([])
            tnum = row[trial_number_column]
            ax.set_title(f'{g}\n{seg}' if r == 0 else '', fontsize=9)
            ax.text(0.04, 0.96, f'#{int(tnum)}' if pd.notna(tnum) else '#?',
                    transform=ax.transAxes, va='top', ha='left', fontsize=7, color='white',
                    bbox=dict(boxstyle='round', fc='black', alpha=0.45, ec='none'))
    hide = '' if show_dropped else '  (dropped hidden)'
    fig.suptitle(f'{mouse}: kept (blue) vs dropped (red) centroid points  '
                 f'(conf < {cutoff:g} = dropped){hide}', y=1.0)
    fig.tight_layout()
    return fig, axes

In [ ]:
# Toggle show_dropped / conf_cutoff to inspect the thresholding. conf_cutoff=None uses
# CONF_THRESHOLD; set it (e.g. 0.8) to preview a stricter cut without changing step v.
fig, axes = figure_per_trial_flagged(
    position, confidence, trials_filtered, ROOT, MOUSE,
    conf_threshold=CONF_THRESHOLD,
    show_dropped=True,      # False -> hide dropped points, see the surviving path
    conf_cutoff=None,       # e.g. 0.8 -> recolour points below 0.8 as dropped (live preview)
    centroid_points=CENTROID_POINTS,
    max_trials=80,
)
plt.show()

## vii | Tracks: kept vs dropped

The x / y / confidence tracks with points coloured kept vs dropped and the threshold drawn on
the confidence panel. Same `show_dropped` / `conf_cutoff` toggles.

In [ ]:
def plot_position_tracks_flagged(centroid, centroid_conf, trials, session, *,
                                 conf_threshold=None, show_dropped=True, conf_cutoff=None,
                                 window_s=60, use_cm=None, root=None, event_prefix='Start Trial',
                                 kept_color='#1f77b4', dropped_color='#d62728', figsize=(14, 6),
                                 session_coord='session', time_coord='Time', space_dim='space'):
    """Per-session x/y/confidence tracks with points coloured kept vs dropped.

    Parameters
    ----------
    centroid : xarray.DataArray
        Tracked point (Time x space) with a `session` coord on Time.
    centroid_conf : xarray.DataArray
        Its per-frame confidence (Time,) with the same `session` coord.
    trials : pandas.DataFrame
        The (filtered) trial table.
    session : str
        Session id to plot.
    conf_threshold : float | None
        Stored drop threshold; None -> CONF_THRESHOLD.
    show_dropped : bool
        Draw the dropped (red) points. Default True.
    conf_cutoff : float | None
        Live override: points below this count as dropped, ignoring conf_threshold.
    window_s : float
        Seconds per panel. Default 60.
    use_cm : bool | None
        Positions in cm (True) or px (False). None -> USE_CM.
    root : str | pathlib.Path | None
        Data root for Start Trial events. None -> ROOT.
    event_prefix : str
        Trial-start event prefix. Default 'Start Trial'.
    kept_color, dropped_color : str
        Point colours.
    figsize : tuple
        Figure size per window.
    session_coord, time_coord, space_dim : str
        Coordinate / dimension names.

    Returns
    -------
    None
        Draws one stacked (x, y, confidence) figure per time window.
    """
    if use_cm is None:
        use_cm = USE_CM
    if root is None:
        root = ROOT
    cutoff = (conf_cutoff if conf_cutoff is not None
              else conf_threshold if conf_threshold is not None else CONF_THRESHOLD)
    c = _session_slice(centroid, session, session_coord=session_coord, time_coord=time_coord)
    q = _session_slice(centroid_conf, session, session_coord=session_coord, time_coord=time_coord)
    if c.sizes.get(time_coord, 0) == 0:
        print(f'no centroid frames for session {session!r}.')
        return
    t = c.coords[time_coord].values
    x = c.sel({space_dim: 'x'}).values.astype(float)
    y = c.sel({space_dim: 'y'}).values.astype(float)
    conf = q.values.astype(float)
    unit = 'px'
    if use_cm:
        s = cm_per_px_for(session)
        if np.isnan(s):
            print(f'session {session!r} has no usable cm/px scale; showing pixels.')
        else:
            x, y, unit = x * s, y * s, 'cm'
    dropped = conf < cutoff
    tr_sess = trials[trials['session'] == session]
    st_ev = _start_trial_times(root, session, event_prefix)
    mouse = tr_sess['mouseID'].iloc[0] if len(tr_sess) else ''
    tmin, tmax = float(np.nanmin(t)), float(np.nanmax(t))
    for w0 in np.arange(tmin, tmax, window_s):
        w1 = w0 + window_s
        m = (t >= w0) & (t < w1)
        fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
        for ax, vals, lab in zip(axes, [x, y, conf], [f'x ({unit})', f'y ({unit})', 'confidence']):
            keptm, dropm = m & ~dropped, m & dropped
            ax.plot(t[m], vals[m], '-', color='0.75', lw=0.5, zorder=1)
            ax.scatter(t[keptm], vals[keptm], s=6, color=kept_color, zorder=2)
            if show_dropped:
                ax.scatter(t[dropm], vals[dropm], s=8, color=dropped_color, zorder=3)
            ax.set_ylabel(lab)
            _overlay_trial_events(ax, tr_sess, st_ev, w0, w1, marker_y=None)
            ax.set_xlim(w0, w1)
        axes[2].axhline(cutoff, ls='--', color=dropped_color, lw=1); axes[2].set_ylim(0, 1.02)
        axes[-1].set_xlabel('Time (s)')
        hide = '' if show_dropped else '  (dropped hidden)'
        axes[0].set_title(f'{mouse}  {session}   |   {w0:.0f}-{w1:.0f}s   '
                          f'kept=blue, dropped=red (conf<{cutoff:g}){hide}', fontsize=8)
        fig.tight_layout(); plt.show()

In [ ]:
plot_position_tracks_flagged(
    centroid, centroid_conf, trials_filtered, SESSION,
    conf_threshold=CONF_THRESHOLD,
    show_dropped=True,     # False -> hide dropped points
    conf_cutoff=None,      # e.g. 0.8 -> preview a stricter cut
    window_s=60,
)

## viii | Interpolated dataset

`interpolate_pose`: drop points below `conf_threshold` (`filter_by_confidence`), then fill the
gaps over time (`interpolate_over_time`), per session. All interpolation args are exposed
(`method`, `max_gap`, ...). Confidence is unchanged, so the drop mask still marks the fills.

In [ ]:
def interpolate_pose(position, confidence, *, conf_threshold=None, method='linear',
                     max_gap=None, print_report=False,
                     session_coord='session', time_coord='Time', **interp_kwargs):
    """Drop low-confidence points then interpolate the pose over time, per session.

    Parameters
    ----------
    position : xarray.DataArray
        Full pose (Time x keypoints x space) with a `session` coord on Time.
    confidence : xarray.DataArray
        Matching confidence (Time x keypoints) with the same `session` coord.
    conf_threshold : float | None
        Points with confidence below this are set to NaN before interpolating
        (`filter_by_confidence`). None -> CONF_THRESHOLD.
    method : str
        Interpolation method passed to `interpolate_over_time` (xarray `interpolate_na`):
        'linear', 'nearest', 'slinear', 'quadratic', 'cubic', 'pchip', 'akima', ... .
    max_gap : int | None
        Longest run of consecutive NaN frames to fill; None = no limit.
    print_report : bool
        Forwarded to both movement calls (NaN counts before/after). Default False.
    session_coord, time_coord : str
        Coordinate / dimension names.
    **interp_kwargs
        Any extra keyword args accepted by `interpolate_over_time` / xarray
        `interpolate_na` (e.g. `fill_value`, `order` for polynomial/spline methods).

    Returns
    -------
    xarray.DataArray
        A new pose array, same shape/coords as `position`, with low-confidence points
        dropped and gaps interpolated per session.
    """
    if conf_threshold is None:
        conf_threshold = CONF_THRESHOLD
    other_dims = [d for d in position.dims if d != time_coord]
    out = position.transpose(time_coord, *other_dims).copy()          # Time first, for idx assignment
    sessions = pd.unique(position.coords[session_coord].values)
    for s in sessions:                                                # never bridge across sessions
        idx = np.flatnonzero(position.coords[session_coord].values == s)
        pos_s = position.isel({time_coord: idx}).rename({time_coord: 'time'})
        conf_s = confidence.isel({time_coord: idx}).rename({time_coord: 'time'})
        kept = mfilt.filter_by_confidence(pos_s, conf_s, threshold=conf_threshold,
                                          print_report=print_report)
        filled = mfilt.interpolate_over_time(kept, method=method, max_gap=max_gap,
                                             print_report=print_report, **interp_kwargs)
        out.values[idx, ...] = filled.transpose('time', *other_dims).values
    return out

In [ ]:
# All interpolate_over_time knobs are exposed here. method/max_gap are the usual two;
# extra kwargs (e.g. order= for 'polynomial'/'spline') pass straight through.
INTERP_METHOD = 'linear'    # 'nearest','slinear','quadratic','cubic','pchip','akima',...
INTERP_MAX_GAP = None       # max consecutive-NaN run (frames) to fill; None = no limit

position_interp = interpolate_pose(
    position, confidence,
    conf_threshold=CONF_THRESHOLD,     # drop below this (step v) before filling
    method=INTERP_METHOD,
    max_gap=INTERP_MAX_GAP,
    print_report=True,                 # NaN counts before/after
)
centroid_interp = centroid_position(position_interp, CENTROID_POINTS)   # rebuilt tracked point
centroid_interp

## ix | Reproduce on the interpolated data

Every view rebuilt on `centroid_interp`, same toggles. Red points now show where interpolation
filled the dropped frames; sanity-check those.

In [ ]:
# figure_per_trial reads result[pose_key]; pass a dict so it uses the interpolated pose
# (confidence is unchanged by interpolation).
result_interp = {'dlc:position': position_interp, 'dlc:confidence': confidence}

# 0) path traces (confidence colourmap) on the interpolated pose
fig, axes = figure_per_trial(result_interp, trials_filtered, ROOT, MOUSE,
                             group_by='training_day', centroid_points=CENTROID_POINTS,
                             video_subdir=VIDEO_SUBDIR, max_trials=80)
plt.show()

# i, ii) x / y / confidence tracks on the interpolated centroid
plot_position_tracks(centroid_interp, centroid_conf, trials_filtered, SESSION, window_s=60)

# iv) locomotion speed on the interpolated centroid (spikes from dropped points removed)
plot_locomotion_speed(centroid_interp, trials_filtered, SESSION, window_s=60, smooth_window=None)

# vi, vii) flagged path traces + tracks on the interpolated data (toggles still apply)
figure_per_trial_flagged(position_interp, confidence, trials_filtered, ROOT, MOUSE,
                         conf_threshold=CONF_THRESHOLD, show_dropped=True, conf_cutoff=None,
                         centroid_points=CENTROID_POINTS, max_trials=80)
plt.show()
plot_position_tracks_flagged(centroid_interp, centroid_conf, trials_filtered, SESSION,
                             conf_threshold=CONF_THRESHOLD, show_dropped=True, conf_cutoff=None,
                             window_s=60)